# Evaluation B — Single-Reviewer Human-Grounded Narrative Validation

This unexecuted notebook is a thin, readable view of canonical Evaluation B artifacts. Missing upstream artifacts are reported as **NOT YET AVAILABLE**. Notebook cells do not construct the human reference, run models, call an API, implement metrics, bootstrap confidence intervals, or recreate figures.

Terminology: the reviewed labels form a **single-reviewer human-grounded narrative reference**. SHERLOC Legacy Keywords remain a separate **silver reference**. Abstain cases are narrative-insufficiency diagnostics, not ordinary all-negative references.

## 1. Setup and canonical-artifact gate

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, SVG, display


def locate_repo_root() -> Path:
    configured = os.environ.get("SHERLOC_REPO_ROOT")
    starts = [Path(configured).expanduser()] if configured else []
    starts.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in starts:
        if (candidate / "src/experiments/18_evaluate_evaluation_b.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate SHERLOC_Case_Analysis. Start Jupyter in the repository "
        "or set SHERLOC_REPO_ROOT."
    )


REPO_ROOT = locate_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "outputs/analysis/evaluation_b"
FIGURE_ROOT = REPO_ROOT / "outputs/figures/evaluation_b"
EXPECTED_ANALYSIS_OUTPUTS = {
    *(f"outputs/analysis/evaluation_b/{name}" for name in (
        "silver_vs_human_summary.csv",
        "silver_vs_human_per_label.csv",
        "silver_vs_human_case_level.csv",
        "auxiliary_silver_vs_human_summary.csv",
        "eval_b_main_results.csv",
        "eval_b_bootstrap_cis.csv",
        "eval_b_family_results.csv",
        "eval_b_per_label_results.csv",
        "eval_b_abstain_results.csv",
        "eval_b_abstain_case_level.csv",
        "eval_b_prediction_breadth.csv",
        "model_silver_vs_human_metric_comparison.csv",
        "human_grounded_case_level_errors.csv",
    )),
    *(f"outputs/figures/evaluation_b/{name}" for name in (
        "figure_b1_human_grounded_core_performance.svg",
        "figure_b2_human_grounded_cpmr.svg",
        "figure_b3_silver_vs_human_model_scores.svg",
        "figure_b4_silver_human_label_proportions.svg",
    )),
    "docs/evaluation_b_human_grounded_report.md",
}
REQUIRED_ANALYSIS_INPUTS = {
    "data/annotations/human_grounded_reference_v1.csv",
    "data/annotations/reliability_sample_100.csv",
    "data/processed/sherloc_benchmark_v1.csv",
    "config/experiments/demo_bank_amp_v1.yaml",
    "outputs/analysis/evaluation_b/eval_b_membership_manifest.json",
    "outputs/analysis/evaluation_b/human_grounded_reference_membership_v1.csv",
    "outputs/analysis/evaluation_b/eval_b_training_exclusion_audit.csv",
    "outputs/analysis/evaluation_b/human_annotation_source_manifest.json",
    "outputs/analysis/evaluation_b/human_annotation_qc_summary.json",
    "outputs/analysis/evaluation_b/evaluation_a_integrity_baseline.json",
    "outputs/models/evaluation_b/m1/run_metadata.json",
    "outputs/models/evaluation_b/m2/run_metadata.json",
    "outputs/logs/evaluation_b/llm/m3_diagnostics.json",
    "outputs/logs/evaluation_b/llm/m4_diagnostics.json",
    "outputs/predictions/evaluation_b/m1/predictions.jsonl",
    "outputs/predictions/evaluation_b/m2/predictions.jsonl",
    "outputs/predictions/evaluation_b/m3/eval_b_predictions.jsonl",
    "outputs/predictions/evaluation_b/m4/eval_b_predictions.jsonl",
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_analysis_manifest(manifest: dict) -> tuple[bool, str]:
    if not manifest:
        return False, "manifest missing"
    if manifest.get("status") != "COMPLETE":
        return False, "manifest status is not COMPLETE"
    inputs = manifest.get("inputs_sha256")
    outputs = manifest.get("outputs_sha256")
    if not isinstance(inputs, dict) or not isinstance(outputs, dict):
        return False, "manifest hash maps are missing"
    missing_inputs = REQUIRED_ANALYSIS_INPUTS - set(inputs)
    missing_outputs = EXPECTED_ANALYSIS_OUTPUTS - set(outputs)
    if missing_inputs or missing_outputs:
        return False, f"manifest is incomplete (inputs={sorted(missing_inputs)}, outputs={sorted(missing_outputs)})"
    for relative, expected in {**inputs, **outputs}.items():
        path = (REPO_ROOT / relative).resolve()
        if not path.is_relative_to(REPO_ROOT) or not path.is_file():
            return False, f"bound artifact missing or outside repository: {relative}"
        if sha256_file(path) != str(expected):
            return False, f"bound artifact hash mismatch: {relative}"
    return True, "all frozen inputs and canonical outputs match"


def load_csv(relative: str, required_columns=()) -> pd.DataFrame:
    """Load a canonical evaluator table without synthesizing missing rows."""
    if relative in EXPECTED_ANALYSIS_OUTPUTS and not CANONICAL_ANALYSIS_READY:
        display(Markdown(f"> **NOT YET AVAILABLE:** canonical artifact gate failed for `{relative}`"))
        return pd.DataFrame(columns=list(required_columns))
    path = REPO_ROOT / relative
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{relative}`"))
        return pd.DataFrame(columns=list(required_columns))
    frame = pd.read_csv(path)
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{relative} is missing required columns: {sorted(missing)}")
    return frame


def load_json(relative: str) -> dict:
    path = REPO_ROOT / relative
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{relative}`"))
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def load_jsonl_inventory(method: str, relative: str, expected_n: int | None) -> dict:
    path = REPO_ROOT / relative
    if not path.is_file():
        return {"method": method, "path": relative, "present": False,
                "available": False, "expected_count": expected_n,
                "row_count": None, "validated_count": None}
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()]
    validated_count = sum(
        str(row.get("status", "")).upper() == "SUCCESS_VALIDATED" for row in rows
    )
    complete = expected_n is not None and len(rows) == expected_n == validated_count
    return {
        "method": method,
        "path": relative,
        "present": True,
        "available": complete,
        "expected_count": expected_n,
        "row_count": len(rows),
        "validated_count": validated_count,
    }


def show_table(frame: pd.DataFrame, *, message="canonical rows are unavailable"):
    if frame.empty:
        display(Markdown(f"> **NOT YET AVAILABLE:** {message}."))
    else:
        display(frame)


def show_figure(filename: str):
    path = FIGURE_ROOT / filename
    if not CANONICAL_ANALYSIS_READY or not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return
    display(SVG(filename=str(path)))


analysis_manifest = load_json(
    "outputs/analysis/evaluation_b/evaluation_b_analysis_manifest.json"
)
CANONICAL_ANALYSIS_READY, analysis_gate_detail = validate_analysis_manifest(analysis_manifest)
analysis_status = "COMPLETE" if CANONICAL_ANALYSIS_READY else "NOT YET AVAILABLE"
display(Markdown(
    f"**Canonical Evaluation B analysis:** `{analysis_status}` — {analysis_gate_detail}"
))


## 2. Immutable annotation source and QC

In [ ]:
source_manifest = load_json(
    "outputs/analysis/evaluation_b/human_annotation_source_manifest.json"
)
qc_summary = load_json(
    "outputs/analysis/evaluation_b/human_annotation_qc_summary.json"
)
qc_report = load_csv(
    "outputs/analysis/evaluation_b/human_annotation_qc_report.csv"
)
if source_manifest and qc_summary:
    source = source_manifest.get("source", {})
    display(pd.DataFrame([{
        "immutable_source": source.get("path"),
        "source_rows": source.get("row_count", source_manifest.get("row_count")),
        "source_sha256": source.get("sha256", source_manifest.get("sha256")),
        "qc_status": qc_summary.get("status"),
        "reviewed_n": qc_summary.get("reviewed_n"),
        "not_reviewed_n": qc_summary.get("not_reviewed_n"),
        "skip_n": qc_summary.get("skip_n"),
        "substantive_n": qc_summary.get("substantive_n"),
        "abstain_n": qc_summary.get("abstain_n"),
        "retained_n": qc_summary.get("retained_n"),
        "blocking_issue_count": qc_summary.get("blocking_issue_count"),
    }]))
else:
    display(Markdown("> **NOT YET AVAILABLE:** source-manifest/QC summary pair."))
show_table(qc_report, message="human annotation QC issue rows are unavailable")


## 3. Human AMP label supports

In [ ]:
per_label = load_csv(
    "outputs/analysis/evaluation_b/eval_b_per_label_results.csv",
    required_columns=("method", "family", "label_id", "support"),
)
if not per_label.empty:
    human_support = per_label.loc[
        per_label["method"].eq("M1"), ["family", "label_id", "support"]
    ].drop_duplicates()
    show_table(human_support, message="human AMP support table is unavailable")
else:
    display(Markdown("> **NOT YET AVAILABLE:** finalized human AMP supports."))


## 4. Silver reference versus human narrative reference

In [ ]:
silver_summary = load_csv(
    "outputs/analysis/evaluation_b/silver_vs_human_summary.csv",
	required_columns=(
	    "family", "n", "substantive_n", "comparable_n",
	    "silver_reference_unavailable_n", "exact_set_concordance", "mean_jaccard",
	    "micro_precision_silver_against_human", "micro_recall_silver_against_human",
	    "micro_f1_silver_against_human", "shared_label_count",
	    "silver_only_label_count", "human_only_label_count",
	),
)
silver_per_label = load_csv(
	"outputs/analysis/evaluation_b/silver_vs_human_per_label.csv",
	required_columns=(
	    "family", "label_id", "n", "substantive_n", "comparable_n",
	    "silver_reference_unavailable_n", "silver_support", "human_support",
	    "shared", "silver_only", "human_only", "raw_agreement",
	),
)
auxiliary_summary = load_csv(
	"outputs/analysis/evaluation_b/auxiliary_silver_vs_human_summary.csv",
	required_columns=(
	    "target", "substantive_n", "comparable_n", "excluded_n",
	    "exact_concordance", "status",
	),
)
show_table(silver_summary, message="silver-versus-human family summary is unavailable")
show_table(silver_per_label, message="silver-versus-human per-label rows are unavailable")
show_table(auxiliary_summary, message="auxiliary descriptive comparison is unavailable")


## 5. Leakage audit and prediction inventory

In [ ]:
leakage = load_csv(
    "outputs/analysis/evaluation_b/eval_b_training_exclusion_audit.csv",
    required_columns=(
        "reliability_case_id", "search_rank", "membership_sha256",
        "removed_from_eval_b_supervised_training",
        "removed_from_eval_b_validation", "removed_from_eval_b_threshold_tuning",
        "removed_from_eval_b_supervised_label_selection",
    ),
)
show_table(leakage, message="supervised leakage-exclusion audit is unavailable")

membership_manifest = load_json(
    "outputs/analysis/evaluation_b/eval_b_membership_manifest.json"
)
retained_n = membership_manifest.get("retained_n") if membership_manifest else None
overlap_n = (
    membership_manifest.get("a1_active_m4_demo_overlap_audit", {}).get("overlap_n")
    if membership_manifest else None
)
m4_expected_n = (
    int(retained_n) - int(overlap_n)
    if retained_n is not None and overlap_n is not None else None
)
prediction_inventory = pd.DataFrame([
    load_jsonl_inventory("M1", "outputs/predictions/evaluation_b/m1/predictions.jsonl", retained_n),
    load_jsonl_inventory("M2", "outputs/predictions/evaluation_b/m2/predictions.jsonl", retained_n),
    load_jsonl_inventory("M3", "outputs/predictions/evaluation_b/m3/eval_b_predictions.jsonl", retained_n),
    load_jsonl_inventory("M4", "outputs/predictions/evaluation_b/m4/eval_b_predictions.jsonl", m4_expected_n),
])
display(prediction_inventory)
if not prediction_inventory["available"].all():
    display(Markdown("> **NOT YET AVAILABLE:** one or more frozen prediction artifacts."))


## 6. Main M1–M4 human-grounded results

All primary rows come from one exact common substantive membership. The notebook displays canonical values and deterministic bootstrap intervals; it does not recompute them.

In [ ]:
main_results = load_csv(
    "outputs/analysis/evaluation_b/eval_b_main_results.csv",
    required_columns=(
        "method", "n", "macro_f1", "micro_f1", "exact_set", "jaccard",
        "macro_supported_label_count", "act_cpmr", "means_cpmr", "purpose_cpmr",
    ),
)
bootstrap_cis = load_csv(
    "outputs/analysis/evaluation_b/eval_b_bootstrap_cis.csv",
    required_columns=("method", "metric", "estimate", "ci_low", "ci_high"),
)
show_table(main_results, message="main Evaluation B results are unavailable")
show_table(bootstrap_cis, message="bootstrap confidence intervals are unavailable")


## 7. CPMR, contained recall, and empty-reference behavior

In [ ]:
family_results = load_csv(
    "outputs/analysis/evaluation_b/eval_b_family_results.csv",
    required_columns=(
        "method", "family", "cpmr", "mean_contained_recall",
        "macro_precision_family", "macro_recall_family", "macro_f1_family",
        "supported_label_count", "nonempty_reference_n",
        "cpmr_nonempty_reference", "empty_reference_n",
        "empty_reference_correct_empty_count", "empty_reference_correct_empty_rate",
    ),
)
show_table(family_results, message="family-level CPMR diagnostics are unavailable")


## 8. Narrative-insufficiency (Abstain) diagnostic

In [ ]:
abstain_results = load_csv(
    "outputs/analysis/evaluation_b/eval_b_abstain_results.csv",
    required_columns=(
        "method", "abstain_n", "all_amp_empty_rate",
        "narrative_insufficiency_safe_rate", "mean_total_predicted_label_count",
        "cases_with_any_predicted_act", "cases_with_any_predicted_means",
        "cases_with_any_predicted_purpose",
    ),
)
abstain_cases = load_csv(
    "outputs/analysis/evaluation_b/eval_b_abstain_case_level.csv"
)
show_table(abstain_results, message="Abstain diagnostics are unavailable")
show_table(abstain_cases, message="Abstain case-level rows are unavailable")


## 9. Prediction breadth and silver-scored versus human-scored behavior

In [ ]:
breadth = load_csv(
    "outputs/analysis/evaluation_b/eval_b_prediction_breadth.csv",
    required_columns=(
        "method", "n", "mean_predicted_act_labels", "mean_predicted_means_labels",
        "mean_predicted_purpose_labels", "mean_total_predicted_labels",
        "mean_total_human_labels", "silver_act_reference_available_n",
        "silver_means_reference_available_n", "silver_purpose_reference_available_n",
        "complete_silver_amp_reference_n", "mean_total_silver_labels",
    ),
)
reference_comparison = load_csv(
    "outputs/analysis/evaluation_b/model_silver_vs_human_metric_comparison.csv",
    required_columns=(
        "method", "metric_scope", "metric", "silver_reference_value",
        "human_grounded_value", "delta_human_minus_silver", "human_primary_n",
        "dual_reference_n", "excluded_incomplete_silver_reference_n",
    ),
)
show_table(breadth, message="prediction-breadth rows are unavailable")
show_table(reference_comparison, message="silver/human model-score deltas are unavailable")


## 10. Core figures

In [ ]:
for figure_name in (
    "figure_b1_human_grounded_core_performance.svg",
    "figure_b2_human_grounded_cpmr.svg",
    "figure_b3_silver_vs_human_model_scores.svg",
    "figure_b4_silver_human_label_proportions.svg",
):
    show_figure(figure_name)


## 11. Canonical case-level audit table

In [ ]:
case_level = load_csv(
    "outputs/analysis/evaluation_b/human_grounded_case_level_errors.csv",
    required_columns=(
        "reliability_case_id", "search_rank", "jurisdiction", "fact_summary",
        "human_act_json", "silver_act_json", "silver_act_reference_available",
        "complete_silver_amp_reference_available", "m1_prediction_json",
        "m2_prediction_json", "m3_prediction_json", "m4_prediction_json",
    ),
)
show_table(case_level, message="canonical case-level audit rows are unavailable")


## 12. Interpretation boundary

Only one human reviewer was available, so reviewer-to-reviewer reliability is unavailable. Silver-only labels are not automatically errors: SHERLOC structured metadata may be broader than information recoverable from a Fact Summary. Abstain results are descriptive insufficiency diagnostics. Small-N differences and auxiliary concordance must not be overinterpreted. Evaluation A remains frozen, and no auxiliary predictive benchmark is run here.